# AI Governance Regulatory and Control Crosswalk — Colab Demo

This notebook drives the same `crosswalk/` package used by `cli.py`, just interactively.

**Run the cells in order.** Step 1 gets the code into this Colab session — use **either** Option A (clone from GitHub, once you've pushed the repo) **or** Option B (upload the zip directly). Don't run both.

## Step 1A — Clone from GitHub (use this once the repo is pushed)

In [ ]:
# Replace with your actual GitHub repo URL once it's pushed

# !git clone https://github.com/YOUR_USERNAME/ai-governance-crosswalk.git

# %cd ai-governance-crosswalk

## Step 1B — Or upload the zip directly (use this for now, before you've pushed to GitHub)

In [ ]:
from google.colab import files

uploaded = files.upload()  # select ai-governance-crosswalk.zip when prompted

In [ ]:
import zipfile



zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:

    z.extractall(".")



%cd ai-governance-crosswalk

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 3 — Load the data

In [ ]:
from crosswalk.loader import load_all

from crosswalk.search import by_concept, by_free_text, concept_matrix_markdown



data = load_all()

print(f"Loaded {len(data.frameworks)} frameworks and {len(data.obligations)} obligations.")

## Step 4 — List the governance concepts

In [ ]:
import pandas as pd



concept_df = pd.DataFrame(

    [(c.id, c.name, c.description.strip()) for c in data.concepts.values()],

    columns=["id", "name", "description"],

).sort_values("name")

concept_df

## Step 5 — Look up a concept across all frameworks

This is the core "crosswalk" behavior: pick a governance concern, see every framework's obligation for it.

In [ ]:
concept_to_check = "human_oversight"  # try: vendor_third_party_risk, bias_fairness, transparency_explainability, etc.



results = by_concept(data, concept_to_check)

for o in results:

    print(f"[{o.framework_name}] {o.citation} — {o.title}")

    print(f"  {o.summary.strip()}\n")

## Step 6 — Free-text search

TF-IDF cosine similarity over title + summary + concept names. No internet or model download required — runs fully offline, which matters since this needs to work the same in Colab as on GitHub Actions CI.

In [ ]:
query = "third party vendor oversight"



scored_results = by_free_text(data, query, top_n=6)

for scored in scored_results:

    o = scored.obligation

    print(f"[{o.framework_name}] {o.citation} — {o.title}  (match: {scored.score:.2f})")

    print(f"  {o.summary.strip()}\n")

## Step 7 — Full concept x framework matrix

In [ ]:
from IPython.display import Markdown, display



display(Markdown(concept_matrix_markdown(data)))

## Step 8 — Run the data-integrity tests

These aren't testing AI behavior — they're testing that the YAML files are internally consistent (no obligation references a concept ID that doesn't exist, every obligation has required fields, etc.).

In [ ]:
!pytest -q

---
**Scope and methodology:** crosswalk mappings identify conceptual alignment across selected governance requirements and controls — they do not imply legal or regulatory equivalence. Notably, the 2026 Interagency Model Risk Management Guidance (SR 26-2 / OCC Bulletin 2026-13 / FDIC FIL-15-2026) explicitly excludes generative AI and agentic AI models from its scope; its principles apply to traditional quantitative models and non-generative, non-agentic AI models. See the README for the full framework list and sourcing notes.